# 4. Feature-Space Drifting

The paper contains one blunt sentence that should reorganise how you think about drifting:

> *we report that we were unable to make our method work on ImageNet without a feature encoder.*

Not "it works worse". It does not work. This notebook explains why, and shows the fix on MNIST where you can watch it
happen in two minutes.

The reason is simple once stated: **the drifting field is only as good as the metric its kernel measures.**
$k(\mathbf{x},\mathbf{y})=\exp(-\|\mathbf{x}-\mathbf{y}\|/\tau)$ with $\|\cdot\|$ the Euclidean distance between raw
images asks "do these two images have similar pixel values", which has almost nothing to do with "do these two
images depict similar things". A field built on a meaningless metric transports samples to a meaningless place.

In [ ]:
import math, time
import torch
import torch.nn as nn
import torch.nn.functional as F
import numpy as np
import matplotlib.pyplot as plt
import torchvision
import torchvision.transforms as T

torch.manual_seed(0)
plt.rcParams["figure.dpi"] = 110
device = "cuda" if torch.cuda.is_available() else "cpu"
print("device:", device)

## 1. Euclidean distance between images is not a similarity

A quick demonstration of the problem, before any drifting.

In [ ]:
ds = torchvision.datasets.MNIST("./data", train=True, download=True)
X = ((ds.data.float() / 255.0) * 2 - 1).to(device)          # [60000, 28, 28] in [-1, 1]
Y = ds.targets.to(device)
by_class = [torch.where(Y == c)[0] for c in range(10)]
print("data:", tuple(X.shape))

def real_batch(n):
    i = torch.randint(0, X.shape[0], (n,), device=device)
    return X[i].view(n, -1), Y[i]

# for a query digit, who are its Euclidean nearest neighbours?
anchor_idx = by_class[3][0]
anchor = X[anchor_idx].view(1, -1)
pool_i = torch.randperm(X.shape[0], device=device)[:8000]
pool, pool_lab = X[pool_i].view(-1, 784), Y[pool_i]
d = torch.cdist(anchor, pool)[0]
nn_idx = d.argsort()[:8]

fig, ax = plt.subplots(1, 9, figsize=(12, 1.7))
ax[0].imshow(X[anchor_idx].cpu(), cmap="gray"); ax[0].set_title("query (3)", fontsize=8); ax[0].axis("off")
for k, j in enumerate(nn_idx):
    ax[k + 1].imshow(pool[j].view(28, 28).cpu(), cmap="gray")
    ax[k + 1].set_title(f"{pool_lab[j].item()}", fontsize=8); ax[k + 1].axis("off")
plt.suptitle("8 Euclidean nearest neighbours in pixel space", y=1.15, fontsize=9)
plt.show()

# how often does the pixel-space nearest neighbour share the label?
qi = torch.randperm(X.shape[0], device=device)[:500]
dq = torch.cdist(X[qi].view(-1, 784), pool)
dq.masked_fill_(qi[:, None] == pool_i[None, :], 1e9)          # exclude the query itself if it is in the pool
same = (pool_lab[dq.argmin(1)] == Y[qi]).float().mean()
print(f"pixel-space 1-NN label agreement: {same:.2%}   (chance = 10%)")

About 94%: on MNIST, pixel distance makes a good *nearest-neighbour* classifier, so the trouble is not that the single
closest image is wrong. The drifting field does not use one neighbour, though. It averages over many (the kernel ESS of
Notebook 3 §7), and in 784 dimensions that average mixes in images that are close in pixels without being alike. MNIST
is also the *easy* case: on natural images pixel distance is dominated by global brightness and background. Either way
the kernel weights in $\mathbf{V}$ end up averaging dissimilar samples, and the result is the classic
kernel-regression artefact: **blur**.

## 2. The formulation

Let $\phi$ be a feature extractor. Move the whole loss into feature space (paper Eq 13):

$$\mathcal{L} = \mathbb{E}\Big[\big\|\phi(\mathbf{x}) - \texttt{sg}\big(\phi(\mathbf{x}) + \mathbf{V}(\phi(\mathbf{x}))\big)\big\|^2\Big],
\qquad \mathbf{x}=f_\theta(\boldsymbol{\epsilon})$$

with $\mathbf{V}$ computed from $\phi(\mathbf{y}^{+})$ and $\phi(\mathbf{y}^{-})$. Gradients flow **through $\phi$** into the
generator; $\phi$ itself is frozen. Extending to many features (scales, spatial locations) just sums the losses (Eq 14):

$$\mathcal{L} = \sum_j \mathbb{E}\Big[\big\|\phi_j(\mathbf{x}) - \texttt{sg}\big(\phi_j(\mathbf{x}) + \mathbf{V}(\phi_j(\mathbf{x}))\big)\big\|^2\Big]$$

Three clarifications that matter:

* **This is not a perceptual loss.** A perceptual loss is $\|\phi(\mathbf{x})-\phi(\mathbf{x}_{\text{target}})\|^2$ and
  needs a *paired* target. Here the target is $\phi(\mathbf{x})+\mathbf{V}(\phi(\mathbf{x}))$ — no pairing exists; the
  loss matches *distributions* $\phi_{\#}q$ and $\phi_{\#}p$.
* **It is orthogonal to latent generation.** $\phi$ is a training-time measuring device; the generator can output
  pixels or VAE latents independently. If the generator is in latent space and $\phi$ is a pixel-space encoder, you run
  the VAE decoder in the loop and backpropagate through it.
* **$\phi$ is not used at inference.** Zero inference cost. This is the same "move the work to training time" move as
  the whole method.

## 3. A self-supervised encoder, trained from scratch

The paper's best encoder is a ResNet-34-shaped **masked autoencoder** trained directly on the generation space
(they call it "latent-MAE"), optionally fine-tuned with a classification head. We build the miniature version: a small
conv encoder + deconv decoder, trained to reconstruct 50%-masked inputs, on MNIST, in about 15 seconds on a GPU (several minutes on a CPU).

Why self-supervised and not, say, a classifier? Because the kernel needs *distance* to mean *similarity* — the same
objective SSL optimises. A classifier's penultimate layer collapses everything within a class, which destroys the
within-class geometry the field needs.

In [ ]:
class Encoder(nn.Module):
    """Multi-scale conv encoder. Returns feature maps at 3 scales (14, 7, 7)."""
    def __init__(self, c=32):
        super().__init__()
        self.b1 = nn.Sequential(nn.Conv2d(1, c, 3, 1, 1), nn.GroupNorm(8, c), nn.SiLU(),
                                nn.Conv2d(c, c, 3, 2, 1), nn.GroupNorm(8, c), nn.SiLU())
        self.b2 = nn.Sequential(nn.Conv2d(c, 2*c, 3, 1, 1), nn.GroupNorm(8, 2*c), nn.SiLU(),
                                nn.Conv2d(2*c, 2*c, 3, 2, 1), nn.GroupNorm(8, 2*c), nn.SiLU())
        self.b3 = nn.Sequential(nn.Conv2d(2*c, 4*c, 3, 1, 1), nn.GroupNorm(8, 4*c), nn.SiLU())
    def forward(self, x):
        f1 = self.b1(x); f2 = self.b2(f1); f3 = self.b3(f2)
        return f1, f2, f3                            # [B,32,14,14], [B,64,7,7], [B,128,7,7]

class Decoder(nn.Module):
    def __init__(self, c=32):
        super().__init__()
        self.net = nn.Sequential(
            nn.ConvTranspose2d(4*c, 2*c, 4, 2, 1), nn.GroupNorm(8, 2*c), nn.SiLU(),
            nn.ConvTranspose2d(2*c, c, 4, 2, 1), nn.GroupNorm(8, c), nn.SiLU(),
            nn.Conv2d(c, 1, 3, 1, 1))
    def forward(self, f): return self.net(f)

def train_mae(steps=3000, N=256, mask_ratio=0.5, seed=0):
    torch.manual_seed(seed)
    enc, dec = Encoder().to(device), Decoder().to(device)
    opt = torch.optim.AdamW(list(enc.parameters()) + list(dec.parameters()), 2e-3)
    t0 = time.time()
    for s in range(steps):
        x = real_batch(N)[0].view(N, 1, 28, 28)
        m = F.interpolate((torch.rand(N, 1, 14, 14, device=device) > mask_ratio).float(),
                          scale_factor=2, mode="nearest")           # zero out 2x2 patches
        rec = dec(enc(x * m)[2])
        loss = (((rec - x) ** 2) * (1 - m)).sum() / ((1 - m).sum() + 1e-6)   # masked regions only
        opt.zero_grad(); loss.backward(); opt.step()
    print(f"masked AE trained in {time.time()-t0:.0f}s, final masked MSE {loss.item():.4f}")
    for p in list(enc.parameters()) + list(dec.parameters()):
        p.requires_grad_(False)
    return enc, dec

enc, dec = train_mae()

In [ ]:
# does the learned metric behave better than pixels?
@torch.no_grad()
def embed(img):
    f1, f2, f3 = enc(img)
    return torch.cat([f3.mean((2, 3)), f3.std((2, 3))], -1)

E_pool = embed(pool.view(-1, 1, 28, 28))
E_q = embed(X[qi].view(-1, 1, 28, 28))
dq = torch.cdist(E_q, E_pool).masked_fill_(qi[:, None] == pool_i[None, :], 1e9)   # same self-exclusion
print(f"feature-space 1-NN label agreement: {(pool_lab[dq.argmin(1)] == Y[qi]).float().mean():.2%}"
      f"   (pixels: {same:.2%},  chance: 10%)")

## 4. Multi-scale, multi-location features (paper A.5)

From each feature map $\phi_j\in\mathbb{R}^{H\times W\times C}$ the paper extracts a *set* of $C$-dimensional vectors
and computes an independent drifting loss for each:

* **(a)** one vector per spatial location — $H\times W$ of them
* **(b)** the global mean and the global std over space — 2 vectors
* **(c)** means and stds over $2\times2$ patches — $2\cdot\frac{HW}{4}$ vectors
* **(d)** means and stds over $4\times4$ patches — $2\cdot\frac{HW}{16}$ vectors

plus the mean of squared values per channel of the encoder *input*. Their ablation: FID 9.58 (a,b) → 9.10 (a–c) →
**8.46** (a–d). Richer feature sets help, and once the encoder has run the extra losses are nearly free — the
distance matrices are tiny compared with the convolutions.

Why do the **std** statistics matter? Because a mean-only description of a feature map cannot distinguish "uniform
texture" from "high-contrast structure with the same average". Matching second moments per region is what forces the
generator to produce actual strokes rather than grey mush.

In [ ]:
def pooled_stats(f, k):
    """means and stds over kxk patches -> two [B, n_patches, C] tensors.
    A remainder that does not fill a patch is cropped (14 -> 12 for k=4, 7 -> 6 for k=2)."""
    B, C, H, W = f.shape
    H, W = H // k * k, W // k * k
    if H == 0 or W == 0:
        return None
    p = f[..., :H, :W].reshape(B, C, H // k, k, W // k, k).permute(0, 2, 4, 1, 3, 5).reshape(B, -1, C, k * k)
    return p.mean(-1), p.std(-1)

def feature_set(encoder, img, level="d"):
    """Return a list of [B, C] feature tensors. `level` in {'b','c','d'} adds progressively more."""
    out = []
    B = img.shape[0]
    out.append((img.view(B, 1, -1) ** 2).mean(-1))              # input second moment per channel
    for f in encoder(img):
        Bf, C, H, W = f.shape
        out.append(f.mean((2, 3)))                              # (b) global mean
        out.append(f.std((2, 3)))                               # (b) global std
        out.append(f.permute(0, 2, 3, 1).reshape(B, -1))        # (a) all locations, concatenated
        if level in ("c", "d"):
            r = pooled_stats(f, 2)                               # (c) -- None only if H or W < 2
            if r is not None:
                out += [r[0].reshape(B, -1), r[1].reshape(B, -1)]
        if level == "d":
            r = pooled_stats(f, 4)
            if r is not None:
                out += [r[0].reshape(B, -1), r[1].reshape(B, -1)]  # (d)
    return out

with torch.no_grad():
    fs = feature_set(enc, X[:4].view(4, 1, 28, 28))
print(f"number of feature tensors: {len(fs)}")
print("dims:", [t.shape[-1] for t in fs])

(For clarity this miniature version concatenates the per-location features into one long vector instead of running
$H\times W$ separate losses. The paper does the latter; the concatenated version is cheaper and captures the same
"local structure must match locally" signal on MNIST.)

## 5. The experiment: raw pixels vs feature space

In [ ]:
def feature_scale(f, fp):
    with torch.no_grad():
        return (torch.cdist(f, fp).mean() / math.sqrt(f.shape[-1])).clamp_min(1e-8)

def compute_V(x, y_pos, y_neg=None, tau=0.2):
    N = x.shape[0]
    self_neg = y_neg is None
    if self_neg:
        y_neg = x
    lp = -torch.cdist(x, y_pos) / tau
    ln = -torch.cdist(x, y_neg) / tau
    if self_neg:
        ln = ln - torch.eye(N, device=x.device) * 1e6
    logit = torch.cat([lp, ln], 1)
    A = torch.sqrt(logit.softmax(-1) * logit.softmax(-2))
    P = y_pos.shape[0]
    Ap, An = A[:, :P], A[:, P:]
    return (Ap * An.sum(1, keepdim=True)) @ y_pos - (An * Ap.sum(1, keepdim=True)) @ y_neg

def drift_term(f, fp, taus=(0.02, 0.05, 0.2), eta=0.05):
    C = f.shape[-1]
    S = feature_scale(f, fp)
    fn, fpn = f / S, fp / S
    with torch.no_grad():
        V = 0.0
        for t in taus:
            v = compute_V(fn, fpn, None, tau=t * math.sqrt(C))
            V = V + v / v.pow(2).mean().sqrt().clamp_min(1e-8)
    return F.mse_loss(fn, (fn + eta * V).detach())

class Gen(nn.Module):
    def __init__(self, d_z=64, h=1024, nl=4):
        super().__init__()
        L, d = [], d_z
        for _ in range(nl):
            L += [nn.Linear(d, h), nn.SiLU()]; d = h
        L += [nn.Linear(d, 784)]
        self.net = nn.Sequential(*L); self.d_z = d_z
    def forward(self, z): return torch.tanh(self.net(z))
    @torch.no_grad()
    def sample(self, n): return self(torch.randn(n, self.d_z, device=device))

def train_gen(mode="pixel", level="d", steps=4000, N=256, P=256, lr=2e-4, seed=0):
    torch.manual_seed(seed)
    g = Gen().to(device)
    opt = torch.optim.AdamW(g.parameters(), lr, betas=(0.9, 0.95), weight_decay=0.01)
    sched = torch.optim.lr_scheduler.CosineAnnealingLR(opt, steps)
    t0 = time.time()
    for s in range(steps):
        x = g(torch.randn(N, g.d_z, device=device))
        y = real_batch(P)[0]
        loss = drift_term(x, y)                              # always keep the pixel term
        if mode == "feature":
            fx = feature_set(enc, x.view(N, 1, 28, 28), level)
            with torch.no_grad():
                fy = feature_set(enc, y.view(P, 1, 28, 28), level)
            for a, b in zip(fx, fy):
                loss = loss + drift_term(a, b)
        opt.zero_grad(); loss.backward()
        torch.nn.utils.clip_grad_norm_(g.parameters(), 2.0)
        opt.step(); sched.step()
    print(f"{mode:8s} level={level}: {time.time()-t0:.0f}s  final loss {loss.item():.3e}")
    return g

g_pixel = train_gen("pixel")
g_feat  = train_gen("feature", level="d")

In [ ]:
def report(g, tag, n=1024):
    s = g.sample(n)
    with torch.no_grad():
        d = torch.cdist(s, X[:8000].view(8000, -1)).min(1).values
        # sharpness proxy: mean squared image gradient
        im = s.view(-1, 1, 28, 28)
        sharp = ((im[..., 1:] - im[..., :-1]) ** 2).mean() + ((im[:, :, 1:] - im[:, :, :-1]) ** 2).mean()
    print(f"{tag:14s} pixel std {s.std():.3f}   NN dist {d.mean():.2f}   sharpness {sharp:.4f}")

Xr = X.view(-1, 784)
with torch.no_grad():
    imr = Xr[:1024].view(-1, 1, 28, 28)
    sharp_r = ((imr[..., 1:] - imr[..., :-1]) ** 2).mean() + ((imr[:, :, 1:] - imr[:, :, :-1]) ** 2).mean()
print(f"{'real data':14s} pixel std {Xr.std():.3f}   NN dist  0.00   sharpness {sharp_r:.4f}")
report(g_pixel, "raw pixels")
report(g_feat,  "feature space")

fig, ax = plt.subplots(3, 10, figsize=(13, 4.1))
for j in range(10):
    ax[0, j].imshow(Xr[j].view(28, 28).cpu(), cmap="gray")
    ax[1, j].imshow(g_pixel.sample(1).view(28, 28).cpu(), cmap="gray")
    ax[2, j].imshow(g_feat.sample(1).view(28, 28).cpu(), cmap="gray")
for i, t in enumerate(["real", "drifting, raw pixels", "drifting, feature space"]):
    ax[i, 0].set_ylabel(t, fontsize=7)
for a in ax.ravel():
    a.set_xticks([]); a.set_yticks([])
plt.tight_layout(); plt.show()

The middle row is the paper's failure mode made visible: **blur**. Raw-pixel drifting produces the kernel-weighted
average of nearby training images, which is a superposition of digits. Its pixel std is well below the data's, and its
sharpness statistic is roughly half. The bottom row, from the same generator architecture with the same drifting
loss but distances measured in the SSL feature space, has real strokes and full contrast. Its sharpness statistic is far above the raw-pixel run's and lands near the data's; it can overshoot, because the statistic also rewards high-frequency texture, so read it alongside the images rather than as a target to match exactly.

Be honest about what this small demo does *not* show: a 4M-parameter MLP with a 15-second (GPU) encoder produces digits
whose *identity* is often ambiguous. That is a capacity and conditioning problem, not a metric problem — Notebook 5
adds class conditioning and CFG, which is what makes the digits identifiable.

## 6. The feature-set ablation

In [ ]:
for level, desc in [("b", "(a,b): locations + global mean/std"),
                    ("c", "(a-c): + 2x2 pooled mean/std"),
                    ("d", "(a-d): + 4x4 pooled mean/std")]:
    g = train_gen("feature", level=level, steps=2500)
    report(g, level)
    print(f"    {desc}")

## 7. Why encoder quality dominates

The paper's Table 3 is worth staring at. Holding the generator fixed at B/2 and 100 epochs, changing only $\phi$:

| encoder | FID |
|---|---|
| SimCLR ResNet-50 | 11.05 |
| MoCo-v2 ResNet-50 | 8.41 |
| latent-MAE, width 256, 192 ep | 8.46 |
| latent-MAE, width 640, 192 ep | 6.30 |
| latent-MAE, width 640, 1280 ep | 4.28 |
| latent-MAE, width 640, 1280 ep, + classifier fine-tune | **3.36** |

A better *measuring device* cut FID by 2.5x with no change to the generator. Their explanation is the one we have
been building toward: the field's strength comes from the kernel resolving similarity, so
*a strong encoder reduces the occurrence of a nearly flat kernel*. In pixel space, in high dimensions, almost every
pair of images is far apart, every kernel evaluation is in the tail, and the drift estimate is starved.

Practically, this reframes drifting: **the generator is the cheap part; the metric is the model.** If you are porting
drifting to a new domain (Notebooks 8–10, and `PROTENIX_DRIFTING.md`), the first question is not "what architecture"
but "what is my $\phi$, and does distance in it mean similarity for my data".

## Summary

* The kernel's metric *is* the model's notion of similarity. Raw high-dimensional spaces do not provide one.
* Moving the loss into a frozen SSL feature space costs nothing at inference and is the difference between
  working and not working.
* It is **not** a perceptual loss: the target is the drifted feature, so no pairing is needed and the objective
  matches distributions in $\phi$-space.
* Use many features — scales, locations, means **and stds** — each with its own normalised drift, summed.
* Encoder quality dominates generator quality. Budget accordingly.

Next: [5. Conditioning and Classifier-Free Guidance](<5. Conditioning and Classifier-Free Guidance.ipynb>).